# 📜 The Math-to-NumPy Protocol (v2.2)
*The Universal Translation Layer (Patched for Deep Physics)*

***Objective:*** A structured framework to convert mathematical expressions into vectorized NumPy code.

### Step 1: The Dependency Check (The Stop Sign)
* **Objective:** Identify Recursion.
* **Look at:** RHS Indices.
* **Action:**
    * *Recursive* ($x_t$ depends on $x_{t-1}$): **STOP**. Use a Loop.
    * *Independent*: Proceed to Vectorization.

### Step 2: The Global Map (The LHS Lock)
* **Objective:** Define the Tensor Universe.
* **Logic:** Distinguish between indices that stay and indices that vanish.
    * **Free Indices (LHS):** These dictate the permanent axes of your final tensor.
    * **Summation Indices (RHS):** Temporary axes used for calculation, then reduced (summed).
* **Example:** $R_{ij} = \sum_k (\dots)$
    * $i \to$ Axis 0
    * $j \to$ Axis 1
    * $k \to$ Axis 2 (to be reduced)

### Step 3: The Shape Classifier (The Branching Path)
* **Objective:** Handle Iteration Limits & Access Patterns.
* **Look at:** The relationship between indices.
* **Action:**
    1.  **Rectangular (Standard):**
        * *Sign:* Sums over full ranges ($\sum_{i=1}^N$).
        * *Tool:* Standard Broadcasting or Slicing (`:`)
    2.  **Irregular (Triangular):**
        * *Sign:* Sum depends on position ($\sum_{j<i}$).
        * *Tool:* **Masking** (`A * np.tril(1)`) or `np.where`.
    3.  **Indirect (Scatter/Mesh):**
        * *Sign:* LHS index is a list/map (e.g., $A[\text{nodes}_i] \leftarrow \dots$).
        * *Logic:* Multiple inputs might map to the same output "bin" (Collisions).
        * *Tool:* **`np.add.at(Target, Indices, Values)`**. (Note: Standard assignment `=` will fail here).

### Step 4: Vector Representation
* **Policy:** Store data as Flat Arrays $(N,)$.
* **Constraint:** Only promote to Column Vectors $(N,1)$ immediately before a strict Matrix Dot Product (`@`).

### Step 5: The Alignment (Explicit Broadcasting)
* **Objective:** Mechanics over Magic.
* **Action:** Insert `None` (or `np.newaxis`) for missing indices to match the **Global Map**.
* **Target:** If Global Map is $(i, j)$ and term has $j$ only $\to$ `Term[None, :]`.

# 🥋 Vectorization Dojo


## Challenge 1
### The Equation: Weighted Interaction Matrix
We need to compute matrix $E$, where each element is an interaction between vector $A$ and vector $B$, normalized by the sum of interactions.

$$E_{ij} = \frac{\exp(A_i \cdot B_j)}{\sum_{k} \exp(A_i \cdot B_k)}$$

**Variables:**
* $A$: Vector of size $N$
* $B$: Vector of size $M$

## Step 1: Dependency Check
**Question:** Is there recursion? Does $E_{ij}$ depend on $E_{i-1, j}$?

**Analysis:** No. All indices ($i, j, k$) are independent.
**Decision:** Proceed to Vectorization.

## Step 2: The Global Map
**Question:** What are the axes of our universe?

**Analysis:**
* **Free Indices (LHS):** $i, j$. These form the output shape.
* **Summation Indices (RHS):** $k$. This axis will be reduced.

**Mapping:**
* $i \to$ Axis 0
* $j \to$ Axis 1

In [ ]:
import numpy as np

# Setup Dummy Data
N = 5
M = 3
A = np.random.rand(N) # Shape (N,)
B = np.random.rand(M) # Shape (M,)

## Step 5: The Alignment & Solution

**Logic:**
1. **Align:** We need $A_i$ (Axis 0) and $B_j$ (Axis 1).
   - $A \to$ `A[:, None]`
   - $B \to$ `B[None, :]`
2. **Numerator:** Compute the interaction grid $(N, M)$.
3. **Denominator:** Compute the interaction grid using $B_k$, sum over $k$ (Axis 1), and **reshape** to maintain alignment.

In [ ]:
# 1. The Numerator
# Map: (i, j) -> (Axis 0, Axis 1)
interaction = A[:, None] * B[None, :]  # Broadcasts to (N, M)
numerator = np.exp(interaction)

# 2. The Denominator
# We sum over k (which sits in Axis 1 position for B)
# Trap: np.sum reduces dimension to (N,). We must keep it (N, 1) for division.
denominator_sum = np.sum(numerator, axis=1) # Result: (N,)
denominator = denominator_sum[:, None]      # Result: (N, 1)

# 3. Final Calculation
# (N, M) / (N, 1) -> Valid Broadcast
E = numerator / denominator

print(f"Shape of A: {A.shape}")
print(f"Shape of B: {B.shape}")
print(f"Shape of Result E: {E.shape}")

Shape of A: (5,)
Shape of B: (3,)
Shape of Result E: (5, 3)


## Challenge 2
### The Equation: Causal Cumulative Signal
We want to calculate a signal $Y$ at time $t$, which is the weighted sum of inputs $X$ from all *previous* times $k$ (up to and including $t$).

$$Y_t = \sum_{k=0}^{t} (W_{tk} \cdot X_k)$$

**Variables:**
* $W$: Weight matrix of shape $(N, N)$ (Time-dependent weights)
* $X$: Input signal vector of size $(N,)$
* $t, k$: Time indices from $0$ to $N-1$

### Protocol Analysis
* **Step 1 (Dependency Check):**
    * **Question:** Is there recursion?
    * **Answer:** No. $Y_t$ does not depend on the *result* of $Y_{t-1}$. It is an independent sum.
    * **Decision:** Vectorize.
* **Step 2 (The Global Map):**
    * **Free Indices:** $t$ (LHS).
    * **Summation Index:** $k$ (RHS).
    * **Map:** $t \to$ Axis 0, $k \to$ Axis 1.
* **Step 3 (Shape Classifier):**
    * **Limit:** $k=0$ to $t$ ($k \le t$).
    * **Type:** Irregular / Triangular.
    * **Action:** We must use **Masking** (`np.tril`) because we are vectorizing the loop over $t$.
* **Step 5 (Alignment - The "Axis Trap"):**
    * **Term:** $X_k$.
    * **Target:** $X$ must align with $k$ (Axis 1) to multiply against $W_{tk}$.
    * **Action:** Reshape $X$ to `(1, N)` using `X[None, :]`. (Note: `X[:, None]` would align to $t$, which is incorrect).

---

In [ ]:
import numpy as np
N =30
W =np.random.rand(N,N)
X =np.random.rand(N,)
T =W[:,:]*X[None,:]
T2 =np.tril(T)
Y =np.sum(T2,axis =1)
print(Y)

[0.56101492 0.39371597 0.63015652 0.64815842 1.26252972 1.95719294
 2.63495951 1.24018195 3.54107134 3.91439347 3.68519525 3.96084038
 3.69482879 5.45810953 4.15399675 4.80090873 5.05757397 6.85269755
 6.02821763 4.20722038 6.06812705 6.32742871 7.36774263 5.42421401
 6.87195824 8.30180117 8.39446353 8.12663293 8.69502018 8.46079946]


## Challenge 2.5 (The Retrial)
### The Equation: The Cumulative Field
We are calculating the total Field Strength $F$ at position $i$. The field at $i$ is the sum of influences from all *previous* positions $j$ (where $j \le i$), weighted by the mass $M$ at that position.

$$F_i = \sum_{j=0}^{i} (G_{ij} \cdot M_j)$$

**Variables:**
* $G$: Interaction Grid of shape `(N, N)` (Row $i$, Col $j$)
* $M$: Mass vector of shape `(N,)`
* $i, j$: Spatial indices from $0$ to $N-1$

### Protocol Analysis
* **Step 2 (The Global Map):**
    * **Free Index:** $i$ (Axis 0).
    * **Summation Index:** $j$ (Axis 1).
* **Step 3 (Shape Classifier):**
    * **Limit:** $j \le i$.
    * **Geometry:** Lower Triangular.
    * **Tool:** `np.tril` (Triangle Lower).
* **Step 5 (Alignment):**
    * **Term:** $M_j$.
    * **Target:** $j$ is Axis 1.
    * **Action:** Reshape $M$ to `(1, N)` using `M[None, :]`.

In [ ]:
import numpy as np
N=30
G =np.random.rand(N,N)
M =np.random.rand(N,)
T =G[:,:]*M[None,:]
T1 =np.tril(T)
F =np.sum(T1,axis=1)
print(F)

[0.5278943  1.08879248 0.28609584 1.14894167 0.98657627 1.69111298
 2.08289168 2.9674951  3.64820939 2.98143163 3.56872697 4.86807951
 3.64075533 3.65219412 3.76188515 5.09090575 4.46570552 6.39068729
 5.02750436 5.26773359 6.56069686 5.7474962  6.10253303 6.68723558
 7.45097915 8.74639484 8.51100085 6.68182216 6.59492617 8.83205622]


## Challenge 3 (The Final Boss)
### The Equation: Pairwise Particle Distances (3D)
We have two sets of particles in 3D space. We want to calculate the squared Euclidean distance between every particle in set $P$ and every particle in set $Q$.

$$D_{ij} = \sum_{k \in \{x,y,z\}} (P_{ik} - Q_{jk})^2$$

**Variables:**
* $P$: Matrix of shape `(N, 3)` (N particles, 3 coords).
* $Q$: Matrix of shape `(M, 3)` (M particles, 3 coords).
* $k$: The coordinate index (0=x, 1=y, 2=z).

### Protocol Analysis
* **Step 2 (The Global Map - High Dimensional):**
    * **Free Indices:** $i$ (Axis 0), $j$ (Axis 1).
    * **Summation Index:** $k$ (Axis 2).
    * **Note:** $k$ exists in the input but is summed out in the output.
* **Step 5 (Alignment):**
    * **Term $P_{ik}$:** Has axes $(i, k)$. Missing $j$ (Axis 1).
        * **Action:** `P[:, None, :]` $\to$ Shape `(N, 1, 3)`.
    * **Term $Q_{jk}$:** Has axes $(j, k)$. Missing $i$ (Axis 0).
        * **Action:** `Q[None, :, :]` $\to$ Shape `(1, M, 3)`.
* **Reduction:**
    * After computing the squared difference `(N, M, 3)`, we sum over $k$ (Axis 2).

In [ ]:
import numpy as np
N =30
M =20
P =np.random.rand(N,3)
Q =np.random.rand(M,3)
T =P[:,None,:]-Q[None,:,:]
T2 =T*T
D=np.sum(T2,axis=2)
print(D.shape)

(30, 20)
